# __ML in der Logistik__
# Exercise 04 - Neural networks

__Goal__: Understand the structure of a neural network, the type of problems that can be solved, the effect of the parameters of the model on the prediction, according to the dataset, build a single-layer perceptron and a multilayer perceptron in Python.


__Data and application (Task 2.1)__: The dataset contains the static information of 4503 trips recorded on 4 routes in North Europa (Rotterdam > Hamburg, Kiel > Gdynia, Bremerhaven > Hamburg, Felixstowe > Rotterdam). We want to build a single-layer perceptron predicting if the trip was direct or made additional stops.

__Data and application (Task 2.2)__: The dataset contains the static information of 735 trips recorded on the route Rotterdam > Hamburg. We want to build a neural network predicting the duration of a trip.



### Contents

* [1 Theory: basics on neural networks](#t1)
    * [1.1 Single-layer perceptron (5 min)](#t11)
    * [1.2 Multilayer perceptron (10 min)](#t12)
    * [1.3 Calculate the output of a neural network (15 min)](#t13)
* [2 Practice: predict additional stops](#t2)
    * [2.0 Plot 2 attributes against each other (5 min)](#t20)
    * [2.1 MLPClassifier algorithm: Simple perceptron (10 min)](#t21)
    * [2.2 MLPClassifier algorithm: Multilayer perceptron (15 min)](#t22)

## 1 Theory: basics on neural networks <a class="anchor" id="t1"></a>

### 1.1 Single-layer perceptron (5 min) <a class="anchor" id="t11"></a>

_Explain the algorithm of perceptron (or single-layer perceptron) for binary classiffcation._

: A single-layer perceptron is one of the simplest types of neural networks. It is used to classify data into two classes (binary classification). It works by computing a weighted sum of the inputs, applying an activation function (usually a step function), and adjusting weights based on errors.

_What kind of problems are solved with a single-layer perceptron algorithm?_

: Linearly separable problems: Single-layer perceptrons can only solve problems where a straight line (or hyperplane in higher dimensions) can separate the two classes.


### 1.2 Multilayer perceptron (10 min) <a class="anchor" id="t12"></a>

_Why would we use a multilayer perceptron instead of a single-layer perceptron?_

: Can solve non-linear problems because it has hidden layers that transform inputs in complex ways.

_Simply explain the concept, the layers and their connections._

: Layers:
Input layer: Receives the features.
Hidden layer(s): Intermediate layers that capture non-linear relationships.
Output layer: Produces the final prediction.
Connections: Every node in one layer is usually connected to every node in the next layer (fully connected).

_What is a multilayer perceptron with zero hidden layer?_

:If there are no hidden layers, the MLP reduces to a single-layer perceptron.

_In a multilayer perceptron: how many nodes does the input layer contain? And the output layer?_

:Input layer nodes: Equal to the number of features in the dataset.
Output layer nodes: Equal to the number of classes (1 for binary classification, >1 for multi-class).

_Quickly explain the effect of the learning rate parameter, in natural language (intuition)._

: Learning rate controls how much we adjust weights at each step.
Intuition:
Too small: Learning is slow, may take many iterations.
Too large: Learning is unstable, may overshoot the minimum.
Just right: Converges efficiently to a good solution.

_What is the backpropagation algorithm? What is it used for?_

What it is: A method to train multilayer perceptrons.
How it works:
Compute output error.
Propagate error backward through the network.
Adjust weights using gradient descent to minimize error.
Purpose: Enables MLPs to learn from mistakes and handle non-linear problems.

### 1.3 Calculate the output of a neural network (15 min) <a class="anchor" id="t13"></a>

We are given a neural network, with 2 input nodes (two attributes), 1 hidden layer composed of 2 neurons
and 2 output nodes (see figure). The weights for each connection are written on the network.

_Calculate the outputs_ $O_1$ _and_ $O_2$ _when_ $I_1 = 0.2$ _and_ $I_2 = 0.05$_, with the activation function for each neuron
of the hidden and output layers:_
$$g(h) = \frac{1}{1 + \exp{-iH}}$$
_where_ $iH$ _represents the input value at the node_ $H$_. The input of the node H is a weighted sum of the outputs from the previous layer._

![text](04-calculate-output.PNG)

__Write the values for the outputs__ $O_1$ __and__ $O_2$ __in the following cell (2 points/output).__

In [ ]:
# Values of the outputs

t13_O1 = 
t13_O2 = 

## 2 Practice: predict additional stops <a class="anchor" id="t2"></a>

### 2.0 Plot 2 attributes against each other (5 min) <a class="anchor" id="t20"></a>

For the next tasks, we will want to visualize a dataset containing 2 attributes. We can simply plot them against each other, each point colored by the class attribute, with the following function. As the code is a bit complicated, it is not asked to understand it. You can just use the function as the example in the next task.

You can copy-paste this function into the next exercises to be able to use it for other tasks.

In [ ]:
# Just execute this code

def plot_2att(df, att1, att2, clas_att):
    import matplotlib.pyplot as plt
    '''
    Function for plotting two attributes colored by a class attribute
    
    df: dataframe, the dataset
    att1, att2: strings, the names of the two attributes to be plotted
    clas_att: string, the name of the attribute used for coloration !! must be categorical or object
    '''
    
    df2 = df.copy()
    
    if df2[clas_att].dtype.name in ['object', 'int64', 'float64']:
        df2[clas_att] = df2[clas_att].astype('category')
    elif df2[clas_att].dtype.name != 'category':
        raise Exception('The class attribute must be categorical or object.')
    
    # list_clas: list of the values of the class attribute (TripID)
    list_clas = df2[clas_att].cat.categories.tolist()

    # Create one list for each clas_att value in the array tuples
    tuples = []
    for i in list_clas:
        tuples.append(
            [df2.loc[df[clas_att] == i][att1].values, df2.loc[df[clas_att] == i][att2].values]
        )

    plt.figure(figsize=(12,8))
    j = 0
    for i in tuples:
        label = str(clas_att) + ' = ' + str(list_clas[j])
        j = j + 1
        plt.plot(np.asarray(i[0]), np.asarray(i[1]), 'x', label = label) # use np.asarray in case one of the attribute is
                                                          # categorical, i[0] would return a category and not an array
    plt.legend(loc = 'upper left')
    title = att1 + ' vs. ' + att2 + ' colored by ' + clas_att
    plt.title(title)

### 2.1 MLPClassifier algorithm: Simple perceptron (10 min) <a class="anchor" id="t21"></a>

For this task, we will work with the following dataset. It contains the distance and the duration of 4503 trips on the 4 usual routes, and the fact that the trip was direct (Direct = 1) or that the ship made additional stops (Direct = 0). We want to build a neural network that predicts the attribute Direct.

In [ ]:
# Import and visualize the dataset

import pandas as pd
df = pd.read_csv('04-neural-network_1.csv')
print(df.head())
print(df.dtypes)

But first, let's take a look at the dataset we are working with, with the function presented in the previous section.

In [ ]:
# How to use the function for plot dataset

df = df # the dataset
att1 = 'TotalDistance' # the first attribute to plot (x)
att2 = 'Duration' # the second attribute to plot (y)
clas_att = 'Direct' # the class attribute

plot_2att(df, att1, att2, clas_att)

Looking at the graph, is the problem linearly separable? The separation doesn't have to be perfect with 100% accuracy, but is it possible to separate most of the datapoints into both classes with only one line?

__Answer this question in the following cell, setting the value to__ ``0`` __if the problem is not linearly separable,__ ``1`` __otherwise. (2 points)__

In [ ]:
# Is the problem linearly separable? (integer)

t21_is_linearly_separable = 

We will be using the algorithm ``MLPClassifier`` from the ``sklearn`` library. We want to first build a __single layer perceptron__ model. Here is the documentation of the ``MLPClassifier``: https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html

Which parameter should we change to build a single layer perceptron?

__Write its name in the following cell, between the ' '. (1 point)__

In [ ]:
# Parameter to change for single layer perceptron (string)

t21_name_single_layer = 

How many hidden layers should hold a single layer perceptron?

__Write the value in the following cell. (2 points)__

In [ ]:
# Number of hidden layers for single layer perceptron (integer)

t21_nb_hidden_layers = 

Now let's build the single layer algorithm.

We want to predict the attribute 'Direct' using all the other attributes.

For training and testing, we will use a 80-20% split.

__Write the code to apply the algorithm. Set the parameter to change, with the value__ ``''`` __do not forget__ ``random_state = 1`` __, and compute the accuracy of the model built.__

Note: we get the same warning message as for the random forest algorithm in the previous exercise, we can mitigate it the same way: ``y_train.values.ravel()``.

In [ ]:
# Apply the MLPClassifier algorithm

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score



To verify that we have the right number of layers, we can print the value of the parameter ``n_layers_``. This parameter includes the input and the output layers.

In [ ]:
# Verify that it is a single layer perceptron

nn.n_layers_

_Is the problem linearly separable?_

_Which are the three conditions for the use of a single-layer perceptron? Are they fulfilled?_

_Does the performance obtained in the last code cell confirm your statement about the linearity of the problem? Why?_

### 2.2 MLPClassifier algorithm: Multilayer perceptron (15 min) <a class="anchor" id="t22"></a>

For this task, we will use a new dataset. It contains, for 735 trips on the route Rotterdam > Hamburg the records of the shiptype, the length of the ship, the mean SOG, the distance covered and the total duration. With this dataset, we want to build a neural network to predict the duration of a trip from the other characteristics.

In [ ]:
# Import and visualize the dataset

import pandas as pd
df = pd.read_csv('04-neural-network_2.csv')
print(df.head())
print(df.dtypes)

We first have to make sure that all the attributes can be processed by our algorithm.

The ``MLPClassifier`` can only process numerical attributes. We might want to use the function ``dummies()`` that we created in the first exercise, if we have some attributes that the algorithm cannot process.

In [ ]:
# Function dummies() from exercise 01

def dummies(df, x, att_name):
    df_enc = df.copy()
    new_x = x.copy()
    
    dummies = pd.get_dummies(df[att_name], drop_first = False)
    dummies = dummies.add_prefix("{}#".format(att_name))
    df_enc.drop(att_name, axis = 1, inplace = True)
    df_enc = df_enc.join(dummies)
    
    if att_name in new_x:
        new_x.remove(att_name)
        new_x = [*new_x, *dummies.columns.values]
    
    return df_enc, new_x

__According to the type of the task, which performance measurement should we compute? The__ ``accuracy_score`` __or the__ ``mean_absolute_error`` __? Set the value of the chosen performance to 1 in the following cell, and of the one we don't use to 0. (1 point).__

In [ ]:
# Performance (integers)

t22_accuracy_score = 
t22_mean_absolute_error = 

We will now apply the MLPClassifier. Let's start with building a single layer perceptron, like we did in the previous task.

__Take your code from the previous task and modify it to apply the same algorithm on this new dataset. Do not forget to use the function__ ``dummies()`` __on the train and test datasets.__

In [ ]:
# Apply the MLPClassifier algorithm to build a single layer perceptron

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error



We get a warning message saying that the optimization hasn't converged. We can set the parameter ``max_iter`` to a higher value than its default one. Increase this parameter until the algorithm converges.

The point here is to build a neural network with hidden layers, to be able to build a non-linear model.

__Build the model, but set the parameter for the number of layers to__ ``(100, )`` __(default value). Save the accuracy.__

Note: change again the value for ``max_iter`` if the algorithm still doesn't converge.

In [ ]:
# Apply the MLPClassifier algorithm with one hidden layer containing 100 neurons.

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error



In a neural network, there are a few parameters that have to be tuned to increase the performance of the model. One of them is the configuration of the hidden layers (number of layers and number of neurons in each layer). In the ``MLPClassifier`` algorithm, this configuration can be modified with the parameter ``hidden_layer_sizes``.

__Try the model with different values for this parameter__ (for example, ``(1,)``, ``(10,10,)``, ``(100,100,100,)``, ... and more) __and try to find a configuration giving a better performance than the default one.__

In [ ]:
# Try different combinations for hidden_layer_sizes



Another parameter that we tune here is the learning rate. In this algorithm, the parameter to change is: ``learning_rate_init``. The default value is ``0.001``.

__Set the parameter__ ``hidden_layer_sizes = (100,100,)`` __and try out a couple of combinations for the learning rate__ (for example, 0.00001, 0.001, 0.1, 1, ... and more). __Try to find which learning rate fits the best this dataset.__

In [ ]:
# Try different combinations for learning_rate_init



_Is the problem linear?_

_Compare the performances of the single layer perceptron and the models with hidden layers._

_What is the best number of layers for this problem? What happens if you add more layers?_

_Which learning rate gives the best performance? What do you conclude about the structure of the dataset from this learning rate?_